# NumCompute: End-to-End Machine Learning Framework

> Developed as part of Assignment 2.1 — Modular Scientific Computing Toolkit

This notebook demonstrates an end-to-end machine learning framework built from scratch using **NumPy**, focusing on vectorized computation, numerical stability, and modular design.

### Key Highlights
- Fully vectorized implementations (NumPy-based)
- Clean and consistent API design (`fit`, `transform`)
- Robust handling of missing values and edge cases
- Scalable to large datasets

### Features Covered
- CSV Data Loading (with missing values)
- Synthetic Dataset Generation (for scalability testing)
- Data Preprocessing (Imputer, Scaling, Encoding)
- Sorting & Searching Algorithms
- Ranking & Percentiles
- Statistical Analysis
- Evaluation Metrics
- Gradient & Jacobian Estimation
- Pipeline Abstraction
- Performance Benchmarking (Vectorized vs Loops)

In [1]:
import numpy as np

from numcompute.io import read_csv
from numcompute.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, Imputer
from numcompute.sort_search import top_k, quickselect, binary_search
from numcompute.rank import rank_with_ties, percentile
from numcompute.stats import RunningStats, histogram, quantiles
from numcompute.metrics import accuracy, precision, recall, f1_score, mse, confusion_matrix, roc_auc
from numcompute.optim import finite_diff_grad, jacobian
from numcompute.pipeline import Pipeline, FeatureUnion
from numcompute.benchmarking import benchmark

## 1. Dataset Preparation

We use two types of datasets:

1. Synthetic dataset — to demonstrate scalability and performance
2. Real CSV dataset — to demonstrate practical data handling with missing values

In [2]:
# Synthetic Dataset
import numpy as np

np.random.seed(42)

X_syn = np.random.rand(100000, 5)
y_syn = np.random.randint(0, 2, 100000)

print("Synthetic Dataset Shape:", X_syn.shape)

Synthetic Dataset Shape: (100000, 5)


In [3]:
# Real Dataset (CSV with missing values)

np.savetxt("real_data.csv",
           np.array([[1, 2, np.nan],
                     [3, np.nan, 6],
                     [7, 8, 9]]),
           delimiter=",")

from numcompute.io import read_csv

X_real = next(read_csv("real_data.csv"))

print("Real Dataset:")
print(X_real)

Real Dataset:
[[ 1.  2. nan]
 [ 3. nan  6.]
 [ 7.  8.  9.]]


## 2. Data Preprocessing

Data is cleaned and transformed using modular, vectorized preprocessing components that follow a consistent `fit`–`transform` API.

- **Imputer** → replaces missing values (NaN handling)
- **StandardScaler** → applies z-score normalization
- **MinMaxScaler** → scales features to a fixed range
- **OneHotEncoder** → encodes categorical variables into numerical format

In [4]:
# ----------------------------
# Pipeline on Real Dataset
# ----------------------------
pipe_real = Pipeline([
    ("imputer", Imputer()),
    ("scaler", StandardScaler())
])

X_real_processed = pipe_real.fit_transform(X_real)

print("Processed Real Data:\n", X_real_processed)


# ----------------------------
# Scaling on Synthetic Dataset
# ----------------------------
scaler_syn = StandardScaler()
X_syn_processed = scaler_syn.fit_transform(X_syn)

print("Processed Synthetic Shape:", X_syn_processed.shape)


# ----------------------------
# MinMax Scaling (Real Dataset)
# ----------------------------
mm = MinMaxScaler().fit(X_real)
print("MinMax Scaled (Real Data):\n", mm.transform(X_real))


# ----------------------------
# OneHot Encoding Example
# ----------------------------
cat = np.array([[1], [2], [1], [3]])

ohe = OneHotEncoder().fit(cat)

print("OneHot Encoded:\n", ohe.transform(cat))

Processed Real Data:
 [[-1.06904497 -1.22474487  0.        ]
 [-0.26726124  0.         -1.22474487]
 [ 1.33630621  1.22474487  1.22474487]]
Processed Synthetic Shape: (100000, 5)
MinMax Scaled (Real Data):
 [[0.         0.                nan]
 [0.33333333        nan 0.        ]
 [1.         1.         1.        ]]
OneHot Encoded:
 [[1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]


## 3. Sorting & Searching

We demonstrate efficient sorting and searching operations using NumPy-based, vectorized implementations:

- **Stable Sorting** → preserves order of equal elements
- **Top-K Selection** → uses partial sorting (`argpartition`) for efficiency
- **Quickselect** → finds k-th smallest element using partitioning
- **Binary Search** → performs efficient lookup in sorted arrays

### Demonstration Across Data Scales

The following examples demonstrate algorithm behavior on:
- Small arrays (for clarity)
- Real dataset (CSV-based)
- Large synthetic dataset (scalability)

In [5]:
from numcompute.sort_search import top_k, quickselect, binary_search, stable_sort
import numpy as np

# ----------------------------
# Example 1: Small Array (Basic Demonstration)
# ----------------------------
arr = np.array([5, 1, 9, 3, 7])

print("Sorted:", stable_sort(arr))
print("Top-3 indices:", top_k(arr, 3))
print("Quickselect (k=2):", quickselect(arr, 2))

sorted_arr = stable_sort(arr)
print("Binary search (7):", binary_search(sorted_arr, 7))


# ----------------------------
# Example 2: Real Dataset (First Column)
# ----------------------------
real_col = X_real[:, 0]

print("\nReal Data Sorted (first column):", stable_sort(real_col))
print("Top-2 indices (Real Data):", top_k(real_col, 2))
print("Binary search (real data value):", binary_search(stable_sort(real_col), real_col[0]))

# ----------------------------
# Example 3: Synthetic Dataset (Large Scale)
# ----------------------------
syn_col = X_syn[:, 0]

top_idx = top_k(syn_col, 5)

print("\nTop-5 indices (Synthetic):", top_idx)
print("Top-5 values (Synthetic):", syn_col[top_idx])

Sorted: [1 3 5 7 9]
Top-3 indices: [2 4 0]
Quickselect (k=2): 5
Binary search (7): 3

Real Data Sorted (first column): [1. 3. 7.]
Top-2 indices (Real Data): [2 1]
Binary search (real data value): 0

Top-5 indices (Synthetic): [65857 83455 70403 99201 55962]
Top-5 values (Synthetic): [0.99999461 0.99994117 0.99992852 0.99992615 0.99991081]


## 4. Ranking & Percentiles

We compute rankings with proper tie handling and derive percentile values using vectorized operations.

- **Ranking** → supports multiple strategies (average, dense, ordinal)
- **Tie Handling** → ensures consistent rank assignment for duplicate values
- **Percentiles** → computed using rank-based normalization

In [6]:
from numcompute.rank import rank_with_ties, percentile, rank
import numpy as np

# ----------------------------
# Example 1: Small Array (Basic Demonstration)
# ----------------------------
x = np.array([10, 20, 20, 40])

print("Ranks (with ties):", rank_with_ties(x))
print("Percentiles:", percentile(x))
print("Dense Rank:", rank(x, method='dense'))


# ----------------------------
# Example 2: Real Dataset (First Column)
# ----------------------------
real_col = X_real[:, 0]

print("\nReal Data Ranks:", rank_with_ties(real_col))
print("Real Data Percentiles:", percentile(real_col))


# ----------------------------
# Example 3: Synthetic Dataset (Large Scale)
# ----------------------------
syn_col = X_syn[:, 0]

print("\nSynthetic Data Percentiles (first 10):", percentile(syn_col)[:10])

Ranks (with ties): [0.  1.5 1.5 3. ]
Percentiles: [0.  0.5 0.5 1. ]
Dense Rank: [0. 1. 1. 2.]

Real Data Ranks: [0. 1. 2.]
Real Data Percentiles: [0.  0.5 1. ]

Synthetic Data Percentiles (first 10): [0.37415374 0.15545155 0.0201302  0.18307183 0.60919609 0.78301783
 0.60475605 0.80640806 0.12215122 0.6599366 ]


## 5. Statistical Analysis

We compute descriptive and distribution statistics using vectorized operations:

- **Mean & Standard Deviation** → summarize central tendency and spread  
- **Histogram** → summarizes data distribution 
- **Quantiles** → compute percentile-based summaries  
- **Streaming Statistics** → compute running mean and variance using Welford’s algorithm  

### Welford’s Algorithm

Used to compute mean and variance incrementally in a numerically stable way.  
It processes data in a single pass and is suitable for large or streaming datasets.

In [7]:
from numcompute.stats import mean, std, histogram, quantiles, RunningStats

# ----------------------------
# Example 1: Processed Real Dataset
# ----------------------------
print("Mean (Real):", mean(X_real_processed, axis=0))
print("Std (Real):", std(X_real_processed, axis=0))

print("Histogram (Real):", histogram(X_real_processed[:, 0], bins=3))
print("Quantiles (Real):", quantiles(X_real_processed[:, 0], [25, 50, 75]))


# ----------------------------
# Example 2: Streaming Statistics (Real Data)
# ----------------------------
rs = RunningStats()
rs.update(X_real_processed[:, 0])

print("Running Mean:", rs.mean)
print("Running Variance:", rs.variance())


# ----------------------------
# Example 3: Synthetic Dataset (Large Scale)
# ----------------------------
print("\nMean (Synthetic):", mean(X_syn_processed, axis=0))
print("Std (Synthetic):", std(X_syn_processed, axis=0))

print("Histogram (Synthetic):", histogram(X_syn_processed[:, 0], bins=5))
print("Quantiles (Synthetic):", quantiles(X_syn_processed[:, 0], [25, 50, 75]))

Mean (Real): [7.40148683e-17 0.00000000e+00 0.00000000e+00]
Std (Real): [1. 1. 1.]
Histogram (Real): (array([2, 0, 1]), array([-1.06904497, -0.26726124,  0.53452248,  1.33630621]))
Quantiles (Real): [-0.6681531  -0.26726124  0.53452248]
Running Mean: 1.1102230246251565e-16
Running Variance: 1.4999999925000003

Mean (Synthetic): [ 4.08187262e-15 -9.78322090e-15 -2.08740533e-14 -2.57786015e-14
  2.35906100e-14]
Std (Synthetic): [1. 1. 1. 1. 1.]
Histogram (Synthetic): (array([19919, 20056, 19778, 20060, 20187]), array([-1.73298323, -1.04145859, -0.34993395,  0.34159069,  1.03311533,
        1.72463997]))
Quantiles (Synthetic): [-0.86692571  0.00187642  0.86794935]


## 6. Evaluation Metrics

We evaluate predictions using standard classification and regression metrics:

- **Accuracy, Precision, Recall, F1-score** → classification performance  
- **MSE (Mean Squared Error)** → regression error  
- **Confusion Matrix** → class-wise prediction analysis  
- **ROC AUC** → ranking-based evaluation using prediction scores  

Metrics are demonstrated on both small examples and large-scale synthetic data.

In [8]:
from numcompute.metrics import accuracy, precision, recall, f1_score, mse, confusion_matrix, roc_auc
import numpy as np

# ----------------------------
# Example 1: Small Dataset
# ----------------------------
y_true = np.array([1, 0, 1, 1])
y_pred = np.array([1, 0, 0, 1])
y_scores = np.array([0.9, 0.2, 0.4, 0.8])

print("Accuracy:", accuracy(y_true, y_pred))
print("Precision:", precision(y_true, y_pred))
print("Recall:", recall(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))
print("MSE:", mse(y_true, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("ROC AUC:", roc_auc(y_true, y_scores))


# ----------------------------
# Example 2: Synthetic Dataset (Large Scale)
# ----------------------------
y_pred_syn = np.random.randint(0, 2, len(y_syn))
y_scores_syn = np.random.rand(len(y_syn))

print("\nAccuracy (Synthetic):", accuracy(y_syn, y_pred_syn))
print("F1 (Synthetic):", f1_score(y_syn, y_pred_syn))
print("ROC AUC (Synthetic):", roc_auc(y_syn, y_scores_syn))

Accuracy: 0.75
Precision: 0.999999995
Recall: 0.6666666644444444
F1: 0.799999992
MSE: 0.25
Confusion Matrix:
 [[1. 0.]
 [1. 2.]]
ROC AUC: 0.0

Accuracy (Synthetic): 0.4962
F1 (Synthetic): 0.49590762142325057
ROC AUC (Synthetic): 0.5001532890022916


## 7. Gradient & Jacobian Estimation

We compute numerical derivatives using finite-difference methods:

- **Gradient** → estimates partial derivatives of scalar functions  
- **Jacobian** → computes derivatives of vector-valued functions  
- Supports both **central** and **forward difference** methods  

These techniques are useful when analytical derivatives are unavailable.

In [9]:
from numcompute.optim import finite_diff_grad, jacobian, grad
import numpy as np

# ----------------------------
# Define Functions
# ----------------------------
def f(x):
    return x[0]**2 + x[1]**2

def F(x):
    return np.array([x[0] + x[1], x[0] * x[1]])

x = np.array([2.0, 3.0])


# ----------------------------
# Gradient Computation
# ----------------------------
print("Gradient (central):", finite_diff_grad(f, x))
print("Gradient (forward):", grad(f, x, method='forward'))


# ----------------------------
# Jacobian Computation
# ----------------------------
print("Jacobian:\n", jacobian(F, x))

Gradient (central): [4. 6.]
Gradient (forward): [4.00001 6.00001]
Jacobian:
 [[1. 1.]
 [3. 2.]]


## 8. Pipeline & FeatureUnion

We demonstrate modular data transformation using:

- **Pipeline** → sequential transformation of data  
- **FeatureUnion** → parallel combination of multiple feature transformations  

These abstractions enable clean, reusable, and scalable workflows similar to scikit-learn.

In [10]:
from numcompute.pipeline import Pipeline, FeatureUnion
from numcompute.preprocessing import StandardScaler, Imputer

# ----------------------------
# Pipeline on Real Dataset
# ----------------------------
pipe = Pipeline([
    ("imputer", Imputer()),
    ("scaler", StandardScaler())
])

X_pipe = pipe.fit_transform(X_real)

print("Pipeline Output:\n", X_pipe)


# ----------------------------
# FeatureUnion on Real Dataset
# ----------------------------
fu = FeatureUnion([
    ("scale1", StandardScaler()),
    ("scale2", StandardScaler())
])

fu.fit(X_real)

print("\nFeatureUnion Output:\n", fu.transform(X_real))


# ----------------------------
# Pipeline on Synthetic Dataset
# ----------------------------
pipe_syn = Pipeline([
    ("scaler", StandardScaler())
])

X_syn_pipe = pipe_syn.fit_transform(X_syn)

print("\nSynthetic Pipeline Shape:", X_syn_pipe.shape)

Pipeline Output:
 [[-1.06904497 -1.22474487  0.        ]
 [-0.26726124  0.         -1.22474487]
 [ 1.33630621  1.22474487  1.22474487]]

FeatureUnion Output:
 [[-1.06904497 -1.                 nan -1.06904497 -1.                 nan]
 [-0.26726124         nan -1.         -0.26726124         nan -1.        ]
 [ 1.33630621  1.          1.          1.33630621  1.          1.        ]]

Synthetic Pipeline Shape: (100000, 5)


## 9. Performance Benchmarking

We compare vectorized NumPy operations with equivalent Python loop-based implementations.

- **Vectorized implementation** → uses NumPy for efficient computation  
- **Loop implementation** → standard Python iteration  

This highlights the performance benefits of vectorization in numerical computing.

In [11]:
import numpy as np
import time

# ----------------------------
# Benchmark Helper
# ----------------------------
def benchmark(func, x):
    start = time.time()
    func(x)
    return time.time() - start


# ----------------------------
# Functions
# ----------------------------
def vectorised(x):
    return np.sum(x**2)

def loop(x):
    s = 0
    for i in x:
        s += i * i
    return s


# ----------------------------
# Large Input
# ----------------------------
x = np.random.rand(1_000_000)


# ----------------------------
# Benchmark Execution
# ----------------------------
t_vec = benchmark(vectorised, x)
t_loop = benchmark(loop, x)


# ----------------------------
# Results
# ----------------------------
print("Vectorised:", t_vec)
print("Loop:", t_loop)

print("\nPerformance Comparison")
print("----------------------")
print(f"Vectorised Time: {t_vec:.6f}s")
print(f"Loop Time:       {t_loop:.6f}s")
print(f"Speedup:         {t_loop / t_vec:.2f}x")

Vectorised: 0.0008058547973632812
Loop: 0.05676603317260742

Performance Comparison
----------------------
Vectorised Time: 0.000806s
Loop Time:       0.056766s
Speedup:         70.44x


## 10. Conclusion

This notebook demonstrates a complete machine learning framework built from scratch using NumPy, covering the full pipeline from data loading to evaluation and optimization.

### Key Highlights
- Modular architecture with reusable components  
- Efficient NumPy-based vectorized implementations  
- Consistent `fit`–`transform` API design  
- End-to-end pipeline execution  

### Key Takeaways
- Vectorization significantly improves computational performance  
- Modular design enables scalability and maintainability  
- Numerical stability and edge-case handling are critical in scientific computing  
- The framework works effectively on both sample CSV data and large synthetic datasets  

---